# Lab 07-1 - Cross-validation and Hyperparameter tuning

This lab covers cross-validation tools for model evaluation and hyperparameter tuning in `sklearn`.

In [ ]:
## Standard Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

# KNN will warn you about future updates, but we'll turn these off
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  

We will continue using the Iowa City Home Sales data (introduced in our previous lab) in our examples:

In [ ]:
## Read IC homes data
ic = pd.read_csv(filepath_or_buffer = "../data/IowaCityHomeSales.csv")

## Train-test split
from sklearn.model_selection import train_test_split
train, test = train_test_split(ic, test_size=0.2, random_state=7)

## Separate the target from the predictors (only using numeric predictors)
train_y = train['sale.amount']
test_y = test['sale.amount']
train_X = train.select_dtypes("number").drop(columns = 'sale.amount')
test_X = test.select_dtypes("number").drop(columns = 'sale.amount')

train_X.head()

## A) Cross-validation and Grid Search

Most of the time we'll want to use cross-validation in conjunction with a pipeline. Given below is a simple pipeline that applies a standardization step followed by a model fitting step:

In [ ]:
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

pipe = Pipeline(steps = [
    ('scaler', StandardScaler()),
    ('model', KNeighborsRegressor())
])

Here we did not modify any of the default values in either step by specifying input arguments to either `StandardScaler()` nor `KNeighborsRegressor()`. This is because we will be supplying our own combinations of tuning parameters stored in a dictionary object.

In [ ]:
from sklearn.preprocessing import RobustScaler, MaxAbsScaler
parms = {
    'scaler': [StandardScaler(), RobustScaler(), MaxAbsScaler()],
    'model__n_neighbors': [10,20,30],
    'model__weights': ['uniform','distance'],
    'model__p': [1,2]
}

So there are a total of $3 \times 3 \times 2 \times 2 = 36$ combinations in this dictionary. Recall that dictionaries are used to store `key:value` pairs. In this example, you should notice that the keys involve:

1. the same names as those given to the steps in our pipeline. In our case either `scaler` or `model`
1. the double underscore is used to reference an argument or parameter within a named step. That is, the name `model__p` references the argument `p` in the step named `model`, or the power parameter for Minkowski distance in `KNeighborsRegressor()`.

After setting up the pipeline and a dictionary of parameters we'd like to search over as above, we can use the `GridSearchCV()` [(documentation)](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) function to find the best combination of tuning parameters as determined using 5-fold cross-validation:

In [ ]:
from sklearn.model_selection import GridSearchCV

# First: Set up CV search grid
grid = GridSearchCV(estimator = pipe, param_grid = parms, cv=5)

# Second: Fit the CV search grid to train_X and train_y.
# i.e. compute the estimated error for each of the 36 possible combinations of parms
grid_res = grid.fit(train_X, train_y)

Then of the $3 \times 3 \times 2 \times 2 = 36$ combinations of parameters defined in `parms`, return the combination that was "best."

In [ ]:
print(grid_res.best_estimator_)

In this example we find that the best performing approach uses

1. a standard scaler
1. $k=10$ neighbors
1. inverse-distance weighting
1. `p=1` i.e. the manhattan distance metric as defined in the [help file](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html) for `KNeighborsRegressor()`

The complete grid search results are accessible through the `cv_results_` attribute. However, these results are stored in a dictionary of `numpy` arrays, so we'll first convert them to a `pandas` DataFrame for easier use. The code given below shows the top five best performing tuning parameter combinations and their cross-validated scores:

In [ ]:
# Take CV results and convert to data frame:
full_grid_results = pd.DataFrame(grid_res.cv_results_)
full_grid_results.head()

In [ ]:
# Narrow down which columns to show:
parms_to_show = ['param_model__n_neighbors', 'param_model__p', 'param_model__weights', 'param_scaler', 'mean_test_score']
full_grid_results = full_grid_results[parms_to_show]

# Sort data frame in descending order of mean test score. 
full_grid_results.sort_values(by = 'mean_test_score', ascending=False).head()

Notice the values of `mean_test_score` in the results shown above do not appear to be in the same units of the outcome variable (recall we're predicting home prices that cost hundreds of thousands of dollars). What is this scoring function? According to the results of my "what is default score for GridSearchCV" search in Google, it seems that for 

- Classification problems ($y$ categorical) the default score function is classification accuracy i.e. the proportion correct
- Regression problems ($y$ numerical) the default score function is $R^2$ which if you recall from SDS 291 is the proportion of the total variance of $y$ explained by your model.

So here 73.1% of the variation in $y$ is explained by the model! Quite a bit!

**A word of caution**: all five top-performing combinations above use $k=10$, which is the *smallest* value of `n_neighbors` we tried (the grid was `[10,20,30]`). When the best value sits at the edge of the grid, the true optimum might lie beyond it, i.e. a $k$ smaller than 10 might do even better. In practice, you would re-run the search with a grid that extends past the edge (e.g. `[3,5,10,20,30]`) to make sure the best value is in the interior of the grid.

## B) Question 1 

Use the documentation for `GridSearchCV()` [found here](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) and the list of predefined model evaluation metrics [found here](https://scikit-learn.org/stable/modules/model_evaluation.html#the-scoring-parameter-defining-model-evaluation-rules) to re-do this parameter search using out-of-sample RMSE as the performance evaluation criteria. In other words, we want to use Root Mean Squared Error as the strategy to evaluate the cross-validated performance of each model (on the held-out validation folds).

Note: sklearn's `mean_test_score` column refers to these held-out validation folds, *not* to the test set we set aside with `train_test_split()`. We won't touch the test set until Section D.

Note: you should know that the character string `passthrough` can be used to skip over a data preparation step. For example, here we also want to consider models *without* using PCA dimension reduction, on top of two where we consider 4 and 8 principal components. So we set `passthrough` as one of the possible choices for the dimension `reducer` step.

In [ ]:
from sklearn.decomposition import PCA

## Pipeline
pipe = Pipeline(steps = [
    ('scaler', StandardScaler()),
    ('reducer', PCA()),
    ('model', KNeighborsRegressor())
])

## Parameters to try
parms = {
    'scaler': [StandardScaler(), RobustScaler(), MaxAbsScaler()],
    'reducer': ['passthrough', PCA(n_components=4), PCA(n_components=8)],
    'model__n_neighbors': [10,20,30]
}

## Results: Notice in the output how dimension reduction via PCA was not used in the best estimator.
grid = GridSearchCV(estimator = pipe, param_grid = parms, cv=5)
grid_res = grid.fit(train_X, train_y)
print(grid_res.best_estimator_)

In [ ]:
# Question 1:



## C) Question 2 

The code below creates a new binary outcome variable that records whether a home was assessed at a higher value than what it sold for. We can use this variable to identify homes whose value was "over-assessed" by the county.

- **Part a)**: Set up a machine learning pipeline that includes a re-scaling step, a normalizing transformation, and a $k$-nearest neighbors classifier **NOT** a $k$-nearest neighbors regressor since our outcome variable $y$ is now categorical.
- **Part b)**: Using `train_X_binary` as your predictors and `train_y_binary` as your new outcome variable you want to predict, use a 5-fold cross-validated grid search with classification accuracy as the scoring metric to find the optimal combination of hyperparameters/steps from the following:
    - Either standardization or robust scaling
    - Either the `yeo-johnson` normalization transformation or no normalizing transformation
    - The number of neighbors being either 10, 15, or 25
    - The classifier using either inverse-distance weighting or uniform weighting
- **Part c)**: Print a sorted data frame showing the top 4 combinations of parameters/steps and their cross-validated accuracy scores.

In [ ]:
## Create the binary outcome variable indicating over-assessment
train_y_binary = (train['assessed'] > train['sale.amount']).astype(int)

## Drop `assessed` from the predictors, since it is used to define the outcome
train_X_binary = train_X.drop(columns = 'assessed')

In [ ]:
# Question 2.a)



In [ ]:
# Question 2.b)



In [ ]:
# Question 2.c)



## D) Using the test set

A final thing to know about the `GridSearchCV()` functions is that they will automatically fit the best estimator to the entire training data set after finding it.

Thus, we can evaluate the performance of the best approach selected by our tuning parameter search on the test set using the `predict()`: 

In [ ]:
## Pipeline
pipe = Pipeline(steps = [
    ('scaler', StandardScaler()),
    ('model', KNeighborsRegressor())
])

## Parameters to try
parms = {
    'scaler': [StandardScaler(), RobustScaler(), MaxAbsScaler()],
    'model__n_neighbors': [10,20,30]
}

## Use .predict() to get predictions on the test set. Note here we combine the
# GridSearchCV(), .fit(), and .predict() steps into a single chain
test_y_hat = GridSearchCV(estimator = pipe, param_grid = parms, cv=5).fit(train_X, train_y).predict(test_X)

# Compare true vs predicted y visually
plt.scatter(x = test_y, y = test_y_hat)
plt.xlabel("True y values")
plt.ylabel("Predicted y values")
plt.title("Comparing true vs predicted values")

## Test set RMSE using the best model/steps (calculated "by hand"):
test_RMSE = np.sqrt(np.average((test_y - test_y_hat)**2))
print(test_RMSE)

## E) Other cross-validation functions

Most of the time we'll want to use cross-validation within the context of a data preparation pipeline. However, occasionally we might want to perform cross-validation using a single model and set of data preparation steps for the purposes of creating data visualizations or doing our own "by hand" calculations (ie: making a results table for a publication, etc.)

The code displayed below demonstrates how to obtain cross-validated predictions, compute their Root Mean Squared Error (RMSE), and visualize the residual errors

In [ ]:
## Functions to perform k-fold CV, LOOC, and cross-validated predictions
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_predict
# Function to compute RMSE
from sklearn.metrics import mean_squared_error

## Initialize CV functions
k_folds = KFold(n_splits = 5)
loo = LeaveOneOut()

## Initialize a knn model
knn_model = KNeighborsRegressor(n_neighbors=13)

In [ ]:
## 1. Get cross-validated predictions y_hat using 5-fold cross-validation
train_y_hat_kfold = cross_val_predict(estimator = knn_model, X = train_X, y = train_y, cv = k_folds)

# Compute RMSE
residual_errors_kfold = train_y - train_y_hat_kfold
SE = residual_errors_kfold**2
MSE = np.mean(SE)
RMSE = np.sqrt(MSE)
print("The estimate of the RMSE on new unseen data using 5 fold CV is " + str(RMSE))

# Plot residual errors
plt.hist(residual_errors_kfold)
plt.title("Histogram of residual errors for 5-fold crossvalidation")
plt.show()

In [ ]:
## 2. Get cross-validated predictions y_hat using leave-one out cross-validation
train_y_hat_loo_pred = cross_val_predict(estimator = knn_model, X = train_X, y = train_y, cv = loo)

# Compute RMSE
residual_errors_loo = train_y - train_y_hat_loo_pred
SE = residual_errors_loo**2
MSE = np.mean(SE)
RMSE = np.sqrt(MSE)
print("The estimate of the RMSE on new unseen data using LOOCV is " + str(RMSE))

# Plot residual errors
plt.hist(residual_errors_loo)
plt.title("Histogram of residual errors for LOOCV crossvalidation")
plt.show()